In [3]:
import sys, os
sys.path.append(os.path.abspath(".."))
from src import preprocess

df = preprocess.load_data(r"C:\Users\dutta\housing\data\raw\kc_house_data_NaN.csv")
df = preprocess.clean_data(df)
df = preprocess.engineer_features(df)

X_train, X_test, y_train, y_test = preprocess.split_data(df)

train_temp = X_train.copy()
train_temp['log_price'] = y_train
zipcode_target_map = train_temp.groupby('zipcode')['log_price'].mean()
X_train['zipcode_encoded'] = X_train['zipcode'].map(zipcode_target_map)
X_test['zipcode_encoded'] = X_test['zipcode'].map(zipcode_target_map)

cluster_target_map = train_temp.groupby('location_cluster')['log_price'].mean()
X_train['cluster_avg_price'] = X_train['location_cluster'].map(cluster_target_map)
X_test['cluster_avg_price'] = X_test['location_cluster'].map(cluster_target_map)

X_train = X_train.drop(columns=['zipcode'])
X_test = X_test.drop(columns=['zipcode'])

preprocessor = preprocess.build_preprocessor(X_train)
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(X_train_processed.shape)
print(X_test_processed.shape)

c:\Users\dutta\housing\src\preprocess.py:192: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['grade_x_sqft_living'] = df['grade'] * df['sqft_living']
c:\Users\dutta\housing\src\preprocess.py:193: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['age_x_renovated'] = df['house_age'] * df['was_renovated']
c:\Users\dutta\housing\src\preprocess.py:202: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at onc

(17290, 60)
(4323, 60)


In [2]:
import pgeocode
nomi = pgeocode.Nominatim('us')
result = nomi.query_postal_code("98101")
print(result)

postal_code            98101
country_code              US
place_name           Seattle
state_name        Washington
state_code                WA
county_name             King
county_code             33.0
community_name           NaN
community_code           NaN
latitude             47.6114
longitude          -122.3305
accuracy                 4.0
Name: 0, dtype: object


In [1]:
print("test")

test


In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score
import numpy as np

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)
}

results = {}

for name, model in models.items():
    cv_scores = cross_val_score(
        model, X_train_processed, y_train,
        cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1
    )
    rmse_scores = -cv_scores
    results[name] = {
        'mean_rmse': rmse_scores.mean(),
        'std_rmse': rmse_scores.std()
    }
    print(f"{name}: RMSE = {rmse_scores.mean():.4f} (+/- {rmse_scores.std():.4f})")

Linear Regression: RMSE = 0.1757 (+/- 0.0014)
Random Forest: RMSE = 0.1710 (+/- 0.0021)
XGBoost: RMSE = 0.1663 (+/- 0.0033)


In [5]:
# Approximate dollar-scale RMSE for each model, for intuition
for name, res in results.items():
    approx_dollar_error = np.expm1(res['mean_rmse']) * np.expm1(y_train.mean())
    print(f"{name}: ~RMSE in log scale = {res['mean_rmse']:.4f}")

Linear Regression: ~RMSE in log scale = 0.1757
Random Forest: ~RMSE in log scale = 0.1710
XGBoost: ~RMSE in log scale = 0.1663


In [6]:
from sklearn.metrics import mean_absolute_error

for name, model in models.items():
    model.fit(X_train_processed, y_train)
    preds_log = model.predict(X_test_processed)
    preds_dollar = np.expm1(preds_log)
    actual_dollar = np.expm1(y_test)
    mae_dollar = mean_absolute_error(actual_dollar, preds_dollar)
    print(f"{name}: Test MAE in real dollars = ${mae_dollar:,.0f}")

Linear Regression: Test MAE in real dollars = $75,884
Random Forest: Test MAE in real dollars = $68,481
XGBoost: Test MAE in real dollars = $66,631
